# 03 - Feature Engineering & Time-Series Preparation
### AI-Powered Predictive Bed Demand Forecasting - Albion Care Network

**Notebook 3 of 7** | Forecasting & Modelling Phase

---

## Objectives

This notebook directly implements the brief's Feature Engineering and Time-Series Preparation
scope, carrying forward every actionable finding from Notebook 02:

1. Build a clean, gap-free daily bed-demand panel per hospital-ward-bed_type, excluding the
   burn-in artefact identified in Notebook 02.
2. Engineer the feature families called out in the project's technology stack: lag features,
   rolling statistics, moving averages, holiday indicators, weekend indicators, seasonal
   features, occupancy ratios, length-of-stay statistics, and bed-utilisation metrics.
3. Turn the two strongest occupancy drivers found in Notebook 02 (admissions and ED arrivals,
   correlation +0.68 / +0.69) and the ward-level bottleneck metric (Section 7.1) into proper
   engineered features, not just descriptive statistics.
4. Encode the elective surgery schedule as a genuine leading indicator (planned in advance, so
   using it forward-looking is not leakage), addressing the "planned future demand" objective.
5. Prepare a supplementary hourly panel for ED arrivals, since the brief asks for hourly, daily,
   and weekly forecasts, and Notebook 02 showed hourly granularity matters far more for ED/
   staffing patterns than for inpatient occupancy.
6. Define forecast targets for both daily (t+1) and weekly (t+7) horizons.
7. Split the data by time (not randomly), preserving the two annual cycles available.
8. Explicitly test for data leakage before handing the panel to Notebook 04.

## Background

Notebook 02 established that: the first 10 days of 2024 are an artefact and must be excluded;
occupancy, admissions, ED arrivals, surgery cancellations, and staffing shortfalls all share a
winter seasonal driver; average occupancy and "time spent above a threshold" tell different
bottleneck stories; and discharge-destination mix is not a same-day occupancy predictor
(so length-of-stay, not discharge destination, is the right lever). Every one of those findings
is translated into a concrete, named feature below.

## Inputs

The six cleaned datasets from Notebook 01 (`data/processed/*_clean.parquet`).

## Outputs

- `daily_feature_panel.parquet` / `.csv`: the primary hospital-ward-bed_type-day panel with all
  engineered features, forecast targets, and a train/validation/test split flag, for Notebook 04.
- `hourly_ed_arrivals_panel.parquet` / `.csv`: a supplementary hourly panel supporting
  hourly-granularity forecasting.
- `feature_dictionary.csv`: documentation of every engineered feature.


In [18]:
# --- Setup & configuration -------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

PROC_DIR = Path('../data/processed')

hospital_ref = pd.read_parquet(PROC_DIR / 'hospital_reference_clean.parquet')
adm = pd.read_parquet(PROC_DIR / 'admissions_discharges_clean.parquet')
bed = pd.read_parquet(PROC_DIR / 'bed_inventory_occupancy_clean.parquet')
ed = pd.read_parquet(PROC_DIR / 'ed_outpatient_arrivals_clean.parquet')
surg = pd.read_parquet(PROC_DIR / 'elective_surgery_schedule_clean.parquet')
staff = pd.read_parquet(PROC_DIR / 'staffing_resource_clean.parquet')

# The burn-in artefact identified in Notebook 02 Section 1: exclude before 2024-01-11 from
# every dataset used to build features, so no engineered feature is ever computed from it.
BURN_IN_END = pd.Timestamp('2024-01-11')

bed = bed[bed['datetime'] >= BURN_IN_END].copy()
adm = adm[adm['admission_datetime'] >= BURN_IN_END].copy()
ed = ed[ed['arrival_datetime'] >= BURN_IN_END].copy()

print('Burn-in period excluded. Remaining date range:')
print('bed_inventory_occupancy:', bed['datetime'].min(), '->', bed['datetime'].max())
print('admissions_discharges:', adm['admission_datetime'].min(), '->', adm['admission_datetime'].max())
print('ed_outpatient_arrivals:', ed['arrival_datetime'].min(), '->', ed['arrival_datetime'].max())


Burn-in period excluded. Remaining date range:
bed_inventory_occupancy: 2024-01-11 00:00:00 -> 2025-12-31 23:00:00
admissions_discharges: 2024-01-11 00:02:00 -> 2026-01-01 03:16:00
ed_outpatient_arrivals: 2024-01-11 00:01:00 -> 2025-12-31 23:59:00


---
## Section 1 - Daily Occupancy Panel (Core Target Series)

The panel grain is one row per (`hospital_id`, `ward`, `bed_type`, `date`). Notebook 02 found
that daily/weekly granularity carries most of the useful signal for inpatient occupancy (hourly
occupancy was nearly flat), so this is the primary modelling grain for Notebooks 04-05.

We also carry forward the **bottleneck metric from Notebook 02 Section 7.1** (share of hours
at or above 85%/90% occupancy), since that ranked wards differently -- and more usefully -- than
mean occupancy alone.


In [19]:
bed['date'] = bed['datetime'].dt.normalize()

daily = (bed.groupby(['hospital_id', 'ward', 'bed_type', 'date'])
         .agg(occupied_beds=('occupied_beds', 'mean'),
              staffed_beds=('staffed_beds', 'mean'),
              total_beds=('total_beds', 'mean'),
              closed_beds=('closed_beds', 'mean'),
              hours_ge_85pct=('occupancy_rate', lambda s: (s >= 0.85).mean()),
              hours_ge_90pct=('occupancy_rate', lambda s: (s >= 0.90).mean()))
         .reset_index())

# Occupancy ratio features, explicitly called out in the brief's feature-engineering list.
daily['occupancy_rate'] = daily['occupied_beds'] / daily['staffed_beds']
daily['available_bed_ratio'] = 1 - daily['occupancy_rate']
daily['closed_bed_ratio'] = daily['closed_beds'] / daily['total_beds']
daily['staffed_bed_ratio'] = daily['staffed_beds'] / daily['total_beds']

daily = daily.sort_values(['hospital_id', 'ward', 'bed_type', 'date']).reset_index(drop=True)

# Completeness check: every hospital-ward-bed_type series should have exactly one row per day.
n_series = daily[['hospital_id', 'ward', 'bed_type']].drop_duplicates().shape[0]
n_days = (daily['date'].max() - daily['date'].min()).days + 1
print(f'Series: {n_series}, days per series: {n_days}, expected rows: {n_series * n_days}, actual rows: {len(daily)}')
display(daily.head())


Series: 40, days per series: 721, expected rows: 28840, actual rows: 28840


,hospital_id,ward,bed_type,date,occupied_beds,staffed_beds,total_beds,closed_beds,hours_ge_85pct,hours_ge_90pct,occupancy_rate,available_bed_ratio,closed_bed_ratio,staffed_bed_ratio
0,HHN-BIR-01,Cardiology Ward,Standard,2024-01-11,25.583333,41.500000,42.0,0.0,0.0,0.0,0.616466,0.383534,0.0,0.988095
1,HHN-BIR-01,Cardiology Ward,Standard,2024-01-12,21.500000,41.416667,42.0,0.0,0.0,0.0,0.519115,0.480885,0.0,0.986111
2,HHN-BIR-01,Cardiology Ward,Standard,2024-01-13,19.000000,41.791667,42.0,0.0,0.0,0.0,0.454636,0.545364,0.0,0.995040
3,HHN-BIR-01,Cardiology Ward,Standard,2024-01-14,16.958333,41.708333,42.0,0.0,0.0,0.0,0.406593,0.593407,0.0,0.993056
4,HHN-BIR-01,Cardiology Ward,Standard,2024-01-15,20.875000,41.875000,42.0,0.0,0.0,0.0,0.498507,0.501493,0.0,0.997024


---
## Section 2 - Calendar & Seasonal Features

Notebook 02 found clear weekly (Tuesday-Friday higher, weekend lower) and annual (winter peak,
summer trough) seasonality. We encode both a categorical/indicator form (for tree-based models,
which split on thresholds naturally) and a cyclical sine/cosine form (for models like LSTM or
linear/statistical methods that benefit from a smooth, continuous representation where e.g.
23 Dec and 2 Jan are recognised as close together, not far apart).


In [20]:
daily['day_of_week'] = daily['date'].dt.dayofweek  # Monday = 0
daily['is_weekend'] = (daily['day_of_week'] >= 5).astype(int)
daily['month'] = daily['date'].dt.month
daily['day_of_year'] = daily['date'].dt.dayofyear
daily['week_of_year'] = daily['date'].dt.isocalendar().week.astype(int)

# Cyclical encodings so the model sees day-of-week and month as continuous cycles, not
# discontinuous integers (day 6 and day 0 are adjacent, not 6 apart).
daily['dow_sin'] = np.sin(2 * np.pi * daily['day_of_week'] / 7)
daily['dow_cos'] = np.cos(2 * np.pi * daily['day_of_week'] / 7)
daily['month_sin'] = np.sin(2 * np.pi * daily['month'] / 12)
daily['month_cos'] = np.cos(2 * np.pi * daily['month'] / 12)

# UK (England & Wales) public holidays covering the dataset's two years. Note: Scotland
# (Horizon Edinburgh) observes some different dates in practice (e.g. St Andrew's Day); we use
# a single England & Wales calendar across all five hospitals as a simplifying assumption,
# flagged here rather than silently applied.
uk_holidays = pd.to_datetime([
    '2024-01-01', '2024-03-29', '2024-04-01', '2024-05-06', '2024-05-27', '2024-08-26',
    '2024-12-25', '2024-12-26',
    '2025-01-01', '2025-04-18', '2025-04-21', '2025-05-05', '2025-05-26', '2025-08-25',
    '2025-12-25', '2025-12-26',
])
daily['is_holiday'] = daily['date'].isin(uk_holidays).astype(int)

print('Weekend days:', daily['is_weekend'].sum(), '| Holiday days:', daily['is_holiday'].sum())
display(daily[['date', 'day_of_week', 'is_weekend', 'month', 'is_holiday', 'dow_sin', 'month_sin']].head())


Weekend days: 8240 | Holiday days: 600


,date,day_of_week,is_weekend,month,is_holiday,dow_sin,month_sin
0,2024-01-11,3,0,1,0,0.433884,0.5
1,2024-01-12,4,0,1,0,-0.433884,0.5
2,2024-01-13,5,1,1,0,-0.974928,0.5
3,2024-01-14,6,1,1,0,-0.781831,0.5
4,2024-01-15,0,0,1,0,0.000000,0.5


---
## Section 3 - Lag Features and Rolling Statistics (Target Series)

**Leakage rule used throughout this notebook:** every rolling/lag feature is computed with an
explicit `shift(1)` (or more) *before* the rolling window is applied, so that the feature
describing day *t* only ever uses information from day *t-1* and earlier. This is the single
most important correctness check for a forecasting feature set: without the `shift`, a rolling
mean would include the current day's own value, silently leaking the answer into the input.


In [21]:
grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])

# Lag features.
for lag in [1, 7, 14, 28]:
    daily[f'occupied_lag_{lag}'] = grp['occupied_beds'].shift(lag)

# Rolling statistics / moving averages, always shifted first to exclude the current day.
for window in [7, 14, 30]:
    daily[f'occupied_roll_mean_{window}'] = grp['occupied_beds'].transform(
        lambda s: s.shift(1).rolling(window).mean())
    daily[f'occupied_roll_std_{window}'] = grp['occupied_beds'].transform(
        lambda s: s.shift(1).rolling(window).std())

# Rolling bottleneck exposure -- the Notebook 02 Section 7.1 metric, as a leakage-safe feature.
daily['bottleneck_roll7_ge90pct'] = grp['hours_ge_90pct'].transform(
    lambda s: s.shift(1).rolling(7).mean())

print('New lag/rolling columns added:')
new_cols = [c for c in daily.columns if 'lag' in c or 'roll' in c]
print(new_cols)
display(daily[['date', 'occupied_beds'] + new_cols[:5]].iloc[25:32])


New lag/rolling columns added:
['occupied_lag_1', 'occupied_lag_7', 'occupied_lag_14', 'occupied_lag_28', 'occupied_roll_mean_7', 'occupied_roll_std_7', 'occupied_roll_mean_14', 'occupied_roll_std_14', 'occupied_roll_mean_30', 'occupied_roll_std_30', 'bottleneck_roll7_ge90pct']


,date,occupied_beds,occupied_lag_1,occupied_lag_7,occupied_lag_14,occupied_lag_28,occupied_roll_mean_7
25,2024-02-05,20.083333,18.958333,15.541667,23.833333,NaN,20.476190
26,2024-02-06,23.083333,20.083333,22.458333,28.666667,NaN,21.125000
27,2024-02-07,24.916667,23.083333,26.875000,32.166667,NaN,21.214286
28,2024-02-08,25.000000,24.916667,24.958333,31.833333,25.583333,20.934524
29,2024-02-09,24.375000,25.000000,17.791667,30.083333,21.500000,20.940476
30,2024-02-10,23.666667,24.375000,16.750000,23.625000,19.000000,21.880952
31,2024-02-11,25.041667,23.666667,18.958333,19.666667,16.958333,22.869048


---
## Section 4 - Admissions-Derived Features

Notebook 02 found admissions to be the strongest single correlate of occupancy (+0.68). We
translate this into daily counts by admission type per ward, plus a rolling length-of-stay
statistic (the mechanism, per Notebook 02, that converts an admission rate into bed-days
actually consumed).


In [22]:
daily_adm = (adm.groupby([adm['hospital_id'], 'ward', adm['admission_datetime'].dt.normalize()])
             .agg(n_admissions=('admission_id', 'count'),
                  n_emergency_admissions=('admission_type', lambda s: (s == 'Emergency').sum()),
                  n_elective_admissions=('admission_type', lambda s: (s == 'Elective').sum()),
                  median_los_hours=('length_of_stay_hours', 'median'))
             .reset_index()
             .rename(columns={'admission_datetime': 'date'}))

daily = daily.merge(daily_adm, on=['hospital_id', 'ward', 'date'], how='left')

# Days with zero admissions in a ward are real (not missing) -- fill the counts with 0.
for col in ['n_admissions', 'n_emergency_admissions', 'n_elective_admissions']:
    daily[col] = daily[col].fillna(0)

grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['n_admissions_lag1'] = grp['n_admissions'].shift(1)
daily['n_admissions_roll7'] = grp['n_admissions'].transform(lambda s: s.shift(1).rolling(7).mean())
daily['n_emergency_admissions_roll7'] = grp['n_emergency_admissions'].transform(
    lambda s: s.shift(1).rolling(7).mean())


**Handling sparse length-of-stay data.** Several wards average only 2-9 admissions per day
(checked below), so many ward-days have zero admissions and therefore no length-of-stay value
to summarise. A naive 30-day rolling median would itself return `NaN` whenever the window
contains too few real observations. We use a forgiving `min_periods` and a ward-level fallback
so the feature stays populated without ever looking at future data.


In [23]:
avg_daily_admissions = adm.groupby(['hospital_id', 'ward']).size() / (
    (adm['admission_datetime'].max() - adm['admission_datetime'].min()).days + 1)
print('Lowest-volume wards (average admissions/day):')
print(avg_daily_admissions.sort_values().head(5))

grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['median_los_roll30'] = grp['median_los_hours'].transform(
    lambda s: s.shift(1).rolling(30, min_periods=5).median())
# Forward-fill any remaining gaps within a series (carries the last known LOS estimate forward).
daily['median_los_roll30'] = grp['median_los_roll30'].transform(lambda s: s.ffill())
# Any still-missing values are at the very start of a series before enough history exists --
# fall back to that ward's own all-time median LOS rather than an unrelated ward's value.
ward_overall_median_los = daily.groupby(['hospital_id', 'ward'])['median_los_hours'].transform('median')
daily['median_los_roll30'] = daily['median_los_roll30'].fillna(ward_overall_median_los)

print('Remaining missing median_los_roll30 values:', daily['median_los_roll30'].isna().sum())


Lowest-volume wards (average admissions/day):
hospital_id  ward         
HHN-EDI-01   Day Case Unit    1.209141
             ICU              1.403047
HHN-BIR-01   Day Case Unit    1.854571
             ICU              1.871191
HHN-MAN-01   Day Case Unit    2.171745
dtype: float64
Remaining missing median_los_roll30 values: 0


---
## Section 5 - ED / Outpatient Arrival Features

ED arrivals were the second-strongest occupancy correlate found in Notebook 02 (+0.69). The
source data does not carry a ward field for ED arrivals (arrivals precede ward assignment), so
these features are computed at hospital level and broadcast to every ward within that hospital,
consistent with the schema difference already documented in Notebook 01 Section 7.


In [24]:
daily_ed = (ed.groupby(['hospital_id', ed['arrival_datetime'].dt.normalize()])
            .agg(ed_arrivals=('arrival_id', 'count'),
                 ed_admitted=('outcome', lambda s: (s == 'Admitted').sum()),
                 ed_high_acuity_arrivals=('triage_category',
                                          lambda s: s.isin(['Emergency', 'Resuscitation']).sum()))
            .reset_index()
            .rename(columns={'arrival_datetime': 'date'}))

daily = daily.merge(daily_ed, on=['hospital_id', 'date'], how='left')

grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['ed_arrivals_lag1'] = grp['ed_arrivals'].shift(1)
daily['ed_arrivals_roll7'] = grp['ed_arrivals'].transform(lambda s: s.shift(1).rolling(7).mean())
daily['ed_high_acuity_arrivals_roll7'] = grp['ed_high_acuity_arrivals'].transform(
    lambda s: s.shift(1).rolling(7).mean())

print('ED-derived features added; missing values (expected 0, hospital-level merge is complete):')
print(daily[['ed_arrivals', 'ed_admitted', 'ed_high_acuity_arrivals']].isna().sum())


ED-derived features added; missing values (expected 0, hospital-level merge is complete):
ed_arrivals                0
ed_admitted                0
ed_high_acuity_arrivals    0
dtype: int64


---
## Section 6 - Elective Surgery Schedule as a Leading Indicator

Unlike every other feature so far, the elective surgery schedule is **known in advance** --
a surgery scheduled for two weeks from now is genuinely visible today, in a real operational
system. Using it as a forward-looking feature is therefore not leakage; it is exactly the kind
of "planned future demand" signal the brief calls out explicitly.

The surgery dataset has no `ward` column, only `specialty` and `bed_type_required`. We map
specialty to ward using the same relationship found in Notebook 01/02's admissions data:


In [25]:
specialty_ward_check = pd.crosstab(adm['specialty'], adm['ward'])
display(specialty_ward_check.loc[specialty_ward_check.sum(axis=1) > 0])


ward,Cardiology Ward,Day Case Unit,General Medicine Ward A,General Medicine Ward B,ICU,Oncology Ward,Orthopaedics Ward A,Orthopaedics Ward B
specialty,,,,,,,,
Cardiology,24251,0,0,0,0,0,0,0
Diagnostics,0,7664,0,0,0,0,0,0
General Medicine,0,0,22546,22952,0,0,0,0
ICU,0,0,0,0,8263,0,0,0
Oncology,0,0,0,0,0,14014,0,0
Orthopaedics,0,0,0,0,0,0,14528,14502


Cardiology, Diagnostics, and Oncology map cleanly to a single ward each (Cardiology Ward,
Day Case Unit, Oncology Ward). Orthopaedics splits roughly evenly between Ward A and Ward B, so
we apply the same scheduled-surgery count to both as a simplifying assumption (documented here,
not hidden). General Medicine and ICU admissions are not driven by elective surgery scheduling
in this dataset, so they receive no surgery-based feature (filled with 0).


In [26]:
specialty_to_wards = {
    'Cardiology': ['Cardiology Ward'],
    'Diagnostics': ['Day Case Unit'],
    'Oncology': ['Oncology Ward'],
    'Orthopaedics': ['Orthopaedics Ward A', 'Orthopaedics Ward B'],
}
mapping_rows = [{'specialty': s, 'ward': w} for s, wards in specialty_to_wards.items() for w in wards]
specialty_ward_map = pd.DataFrame(mapping_rows)
display(specialty_ward_map)

daily_surg = (surg.groupby(['hospital_id', 'specialty', 'surgery_date'])
              .size().rename('n_scheduled').reset_index())
daily_surg = daily_surg.merge(specialty_ward_map, on='specialty', how='inner')
daily_surg = (daily_surg.groupby(['hospital_id', 'ward', 'surgery_date'])['n_scheduled']
              .sum().reset_index().rename(columns={'surgery_date': 'date'}))

daily = daily.merge(daily_surg, on=['hospital_id', 'ward', 'date'], how='left')
daily['n_scheduled'] = daily['n_scheduled'].fillna(0)

# Forward-looking window: surgeries scheduled over the NEXT 7 days from day t (t+1 .. t+7).
# A plain shift(-1).rolling(7) would still compute a TRAILING window (just shifted), which is
# the wrong direction; to correctly sum the next 7 days we reverse the series, take a normal
# backward-looking rolling sum on the reversed order (which is forward-looking in real time),
# then reverse back. This is the forward-looking mirror of the shift(1) used for past features.
def forward_rolling_sum(series, window):
    return series[::-1].shift(1).rolling(window).sum()[::-1]

daily = daily.sort_values(['hospital_id', 'ward', 'bed_type', 'date'])
grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['surgeries_scheduled_next_7d'] = grp['n_scheduled'].transform(
    lambda s: forward_rolling_sum(s, 7))

print('Wards receiving a surgery-based feature:', sorted(daily.loc[daily['n_scheduled'] > 0, 'ward'].unique()))
print('Wards with no elective-surgery mapping (feature = 0):',
      sorted(set(daily['ward'].unique()) - set(daily.loc[daily['n_scheduled'] > 0, 'ward'].unique())))


,specialty,ward
0,Cardiology,Cardiology Ward
1,Diagnostics,Day Case Unit
2,Oncology,Oncology Ward
3,Orthopaedics,Orthopaedics Ward A
4,Orthopaedics,Orthopaedics Ward B


Wards receiving a surgery-based feature: ['Cardiology Ward', 'Day Case Unit', 'Oncology Ward', 'Orthopaedics Ward A', 'Orthopaedics Ward B']
Wards with no elective-surgery mapping (feature = 0): ['General Medicine Ward A', 'General Medicine Ward B', 'ICU']


---
## Section 7 - Staffing-Derived Features

Notebook 02 found safe-staffing-ratio compliance to be the strongest staffing-side correlate of
occupancy (-0.26). Staffing rosters are typically finalised a little ahead of the shift itself,
but to stay conservative and avoid any risk of using same-day information not truly available
at forecast time, we lag this feature by one day.


In [27]:
daily_staff = (staff.groupby(['hospital_id', 'ward', 'date'])['safe_ratio_met']
               .apply(lambda s: (s == 'Yes').mean())
               .rename('safe_staffing_rate').reset_index())

daily = daily.merge(daily_staff, on=['hospital_id', 'ward', 'date'], how='left')

grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['safe_staffing_rate_lag1'] = grp['safe_staffing_rate'].shift(1)
daily['safe_staffing_rate_roll7_lag1'] = grp['safe_staffing_rate'].transform(
    lambda s: s.shift(1).rolling(7).mean())

print('Missing safe_staffing_rate (raw, pre-lag):', daily['safe_staffing_rate'].isna().sum())


Missing safe_staffing_rate (raw, pre-lag): 0


---
## Section 8 - Forecast Targets

The brief asks for hourly, daily, and weekly bed-demand forecasts. Per the Notebook 02 finding
that daily/weekly granularity carries the useful inpatient-occupancy signal, this panel defines
two direct-forecasting targets from the daily series: next-day (t+1) and next-week (t+7)
occupied beds. Both are produced by shifting *forward*, the mirror image of the lag features in
Section 3, and by construction they are only `NaN` for the final 1 or 7 days of each series
(where the real future value does not yet exist in the dataset).


In [28]:
grp = daily.groupby(['hospital_id', 'ward', 'bed_type'])
daily['target_next_day_occupied'] = grp['occupied_beds'].shift(-1)
daily['target_next_week_occupied'] = grp['occupied_beds'].shift(-7)

print('Rows with no next-day target (end of series, expected 40):', daily['target_next_day_occupied'].isna().sum())
print('Rows with no next-week target (end of series, expected 280):', daily['target_next_week_occupied'].isna().sum())


Rows with no next-day target (end of series, expected 40): 40
Rows with no next-week target (end of series, expected 280): 280


---
## Section 9 - Supplementary Hourly Panel (ED Arrivals)

Notebook 02 showed inpatient occupancy is nearly flat by hour of day, but ED arrivals have a
strong, clinically meaningful intraday pattern (peaking mid-morning, troughing overnight). This
supplementary hourly panel supports hourly-granularity forecasting for ED/staffing planning,
fulfilling that part of the brief without forcing the (uninformative) hourly grain onto the
inpatient occupancy model.


In [29]:
ed_hourly = (ed.groupby(['hospital_id', ed['arrival_datetime'].dt.floor('h')])
             .size().rename('ed_arrivals').reset_index()
             .rename(columns={'arrival_datetime': 'datetime'}))

# Fill any hospital-hour combinations with zero arrivals (a genuine, not missing, quiet hour).
full_index = pd.MultiIndex.from_product(
    [ed_hourly['hospital_id'].unique(),
     pd.date_range(ed_hourly['datetime'].min(), ed_hourly['datetime'].max(), freq='h')],
    names=['hospital_id', 'datetime'])
ed_hourly = (ed_hourly.set_index(['hospital_id', 'datetime'])
             .reindex(full_index, fill_value=0).reset_index())

ed_hourly['hour'] = ed_hourly['datetime'].dt.hour
ed_hourly['day_of_week'] = ed_hourly['datetime'].dt.dayofweek
ed_hourly['is_weekend'] = (ed_hourly['day_of_week'] >= 5).astype(int)
ed_hourly['hour_sin'] = np.sin(2 * np.pi * ed_hourly['hour'] / 24)
ed_hourly['hour_cos'] = np.cos(2 * np.pi * ed_hourly['hour'] / 24)

grp_h = ed_hourly.groupby('hospital_id')
for lag in [1, 24, 168]:  # 1 hour, 1 day, 1 week
    ed_hourly[f'ed_arrivals_lag_{lag}h'] = grp_h['ed_arrivals'].shift(lag)
ed_hourly['ed_arrivals_roll_mean_24h'] = grp_h['ed_arrivals'].transform(
    lambda s: s.shift(1).rolling(24).mean())
ed_hourly['target_next_hour_arrivals'] = grp_h['ed_arrivals'].shift(-1)

print('Hourly ED panel shape:', ed_hourly.shape)
display(ed_hourly.head())


Hourly ED panel shape: (86520, 13)


,hospital_id,datetime,ed_arrivals,hour,day_of_week,is_weekend,hour_sin,hour_cos,ed_arrivals_lag_1h,ed_arrivals_lag_24h,ed_arrivals_lag_168h,ed_arrivals_roll_mean_24h,target_next_hour_arrivals
0,HHN-BIR-01,2024-01-11 00:00:00,1,0,3,0,0.000000,1.000000,NaN,NaN,NaN,NaN,2.0
1,HHN-BIR-01,2024-01-11 01:00:00,2,1,3,0,0.258819,0.965926,1.0,NaN,NaN,NaN,0.0
2,HHN-BIR-01,2024-01-11 02:00:00,0,2,3,0,0.500000,0.866025,2.0,NaN,NaN,NaN,0.0
3,HHN-BIR-01,2024-01-11 03:00:00,0,3,3,0,0.707107,0.707107,0.0,NaN,NaN,NaN,1.0
4,HHN-BIR-01,2024-01-11 04:00:00,1,4,3,0,0.866025,0.500000,0.0,NaN,NaN,NaN,0.0


---
## Section 10 - Categorical Encoding Strategy

The brief's model shortlist spans tree-based models (XGBoost, LightGBM, CatBoost, Random
Forest), which handle categorical splits natively or via built-in encoders, and other
approaches (SARIMA/SARIMAX, Prophet, LSTM) that need purely numeric input. Rather than
pre-baking a single encoding into this shared panel, `hospital_id`, `ward`, and `bed_type` are
kept as explicit categorical columns here, and Notebook 04 will apply the encoding each model
family actually needs (native categorical handling for tree models; one-hot or embeddings for
models that require it). This avoids locking every downstream model into one encoding choice.


In [30]:
for col in ['hospital_id', 'ward', 'bed_type']:
    daily[col] = daily[col].astype('category')

print(daily[['hospital_id', 'ward', 'bed_type']].dtypes)


hospital_id    category
ward           category
bed_type       category
dtype: object


---
## Section 11 - Time-Based Train / Validation / Test Split

Splitting randomly would leak future information into training (a model could learn from a
day that comes chronologically after a test-set day). Instead we split purely by date,
preserving both annual cycles for training and holding out the most recent quarters for
validation and testing, exactly the discipline needed for a genuine forecasting evaluation.


In [31]:
TRAIN_END = pd.Timestamp('2025-06-30')
VAL_END = pd.Timestamp('2025-09-30')
# TEST runs to the last date for which a 7-day-ahead target still exists in the data.
TEST_END = daily['date'].max() - pd.Timedelta(days=7)

def assign_split(d):
    if d <= TRAIN_END:
        return 'train'
    elif d <= VAL_END:
        return 'validation'
    elif d <= TEST_END:
        return 'test'
    else:
        return 'future_unscored'

daily['split'] = daily['date'].apply(assign_split)
print(daily['split'].value_counts())
print()
print(daily.groupby('split')['date'].agg(['min', 'max']))


split
train              21480
validation          3680
test                3400
future_unscored      280
Name: count, dtype: int64

                       min        max
split                                
future_unscored 2025-12-25 2025-12-31
test            2025-10-01 2025-12-24
train           2024-01-11 2025-06-30
validation      2025-07-01 2025-09-30


**Rationale.** Training covers roughly 16.5 months (all of the first annual cycle plus
half of the second), so every seasonal pattern identified in Notebook 02 is represented at
least once before evaluation. Validation and test each cover a distinct 3-month block from the
second year, so model selection (validation) and final evaluation (test) are on genuinely
unseen, chronologically later periods. The small `future_unscored` remainder has no real target
yet (it needs 7 more days of data than currently exist) and is kept only for completeness, not
for training or evaluation.


---
## Section 12 - Data Leakage Validation

Before handing this panel to Notebook 04, we run explicit checks that every past-looking
feature only used data strictly before its row's date, and that the forward-looking surgery
feature behaves as intended.


In [32]:
# Check 1: occupied_lag_1 for a given row should equal occupied_beds from exactly 1 day earlier
# in the same series.
sample = daily[(daily['hospital_id'] == 'HHN-LON-01') & (daily['ward'] == 'ICU') &
               (daily['bed_type'] == 'Critical Care')].sort_values('date').reset_index(drop=True)
check = (sample['occupied_lag_1'].iloc[1:].reset_index(drop=True) ==
         sample['occupied_beds'].iloc[:-1].reset_index(drop=True))
print('occupied_lag_1 matches prior-day occupied_beds for every row:', check.all())

# Check 2: rolling mean/std at row t must never include row t's own occupied_beds value.
# We verify this by confirming the rolling feature is unaffected by perturbing the current row.
probe = sample.copy()
original_value = probe.loc[10, 'occupied_beds']
probe.loc[10, 'occupied_beds'] = original_value + 10_000  # inject an obviously-wrong value
recomputed_roll7 = probe['occupied_beds'].shift(1).rolling(7).mean()
print('Rolling mean at the perturbed row itself is unaffected by its own (shifted-out) value:',
      recomputed_roll7.iloc[10] == sample['occupied_roll_mean_7'].iloc[10])
print('Rolling mean at row t+1..t+6 SHOULD change (confirms the window is real, not a no-op):')
print((recomputed_roll7.iloc[11:17] != sample['occupied_roll_mean_7'].iloc[11:17]).any())

# Check 3: the forward-looking surgery feature at row t should equal the sum of n_scheduled
# over the next 7 calendar days, not including day t itself.
sample_surg = daily[(daily['hospital_id'] == 'HHN-LON-01') & (daily['ward'] == 'Cardiology Ward')
                     ].sort_values('date').reset_index(drop=True)
row = 50
expected = sample_surg['n_scheduled'].iloc[row + 1: row + 8].sum()
actual = sample_surg['surgeries_scheduled_next_7d'].iloc[row]
print(f'surgeries_scheduled_next_7d matches manual forward-sum at a sample row: {expected == actual}')

# Check 4: target_next_day_occupied at row t must equal occupied_beds at row t+1.
check_target = (sample['target_next_day_occupied'].iloc[:-1].reset_index(drop=True) ==
                 sample['occupied_beds'].iloc[1:].reset_index(drop=True))
print('target_next_day_occupied matches next-day occupied_beds for every row:', check_target.all())


occupied_lag_1 matches prior-day occupied_beds for every row: True
Rolling mean at the perturbed row itself is unaffected by its own (shifted-out) value: True
Rolling mean at row t+1..t+6 SHOULD change (confirms the window is real, not a no-op):
True
surgeries_scheduled_next_7d matches manual forward-sum at a sample row: True
target_next_day_occupied matches next-day occupied_beds for every row: True


All four checks pass. This confirms: lag and rolling features are strictly backward-looking,
the forward-looking surgery feature genuinely sums future scheduled surgeries (as intended, since
that information is legitimately known in advance), and the forecast targets are correctly
aligned to the day they are meant to predict.


---
## Section 13 - Save Outputs


In [33]:
# Drop rows that don't yet have full feature history (before all lag/rolling windows are
# populated) -- this is a "feature-readiness" cutoff, distinct from the burn-in exclusion.
FEATURE_READY_START = BURN_IN_END + pd.Timedelta(days=31)
daily_export = daily[daily['date'] >= FEATURE_READY_START].copy()

print(f'Feature-ready panel: {len(daily_export):,} rows '
      f'({daily.shape[0] - daily_export.shape[0]:,} rows dropped for incomplete feature history)')

daily_export.to_parquet(PROC_DIR / 'daily_feature_panel.parquet', index=False)
daily_export.to_csv(PROC_DIR / 'daily_feature_panel.csv', index=False)

ed_hourly.to_parquet(PROC_DIR / 'hourly_ed_arrivals_panel.parquet', index=False)
ed_hourly.to_csv(PROC_DIR / 'hourly_ed_arrivals_panel.csv', index=False)

print('Saved daily_feature_panel and hourly_ed_arrivals_panel to', PROC_DIR.resolve())


Feature-ready panel: 27,600 rows (1,240 rows dropped for incomplete feature history)
Saved daily_feature_panel and hourly_ed_arrivals_panel to C:\Users\ifech\OneDrive\Desktop\hospital_bed_occupancy_forecast\hospital_bed_occupancy_forecast\data\processed


In [34]:
feature_dictionary = pd.DataFrame([
    ('occupied_beds', 'Target series: mean hourly occupied beds for the day (ward level)'),
    ('occupancy_rate', 'occupied_beds / staffed_beds'),
    ('available_bed_ratio', '1 - occupancy_rate'),
    ('closed_bed_ratio', 'closed_beds / total_beds'),
    ('staffed_bed_ratio', 'staffed_beds / total_beds'),
    ('hours_ge_85pct / hours_ge_90pct', 'Share of hours in the day at/above 85%/90% occupancy (bottleneck exposure)'),
    ('day_of_week / is_weekend / month / day_of_year / week_of_year', 'Calendar features'),
    ('dow_sin / dow_cos / month_sin / month_cos', 'Cyclical encodings of day-of-week and month'),
    ('is_holiday', 'UK (England & Wales) public holiday indicator'),
    ('occupied_lag_1/7/14/28', 'Occupied beds 1/7/14/28 days earlier in the same series'),
    ('occupied_roll_mean/std_7/14/30', 'Rolling mean/std of occupied_beds over the trailing 7/14/30 days (shifted to exclude the current day)'),
    ('bottleneck_roll7_ge90pct', 'Rolling 7-day average of hours_ge_90pct (shifted)'),
    ('n_admissions / n_emergency_admissions / n_elective_admissions', 'Daily admission counts by type, same ward'),
    ('n_admissions_lag1 / n_admissions_roll7', 'Lagged/rolling admission-count features'),
    ('median_los_roll30', 'Rolling 30-day median length of stay (hours), forward-filled and ward-median-backfilled where sparse'),
    ('ed_arrivals / ed_admitted / ed_high_acuity_arrivals', 'Daily ED arrival counts, hospital level'),
    ('ed_arrivals_lag1 / ed_arrivals_roll7 / ed_high_acuity_arrivals_roll7', 'Lagged/rolling ED arrival features'),
    ('n_scheduled', 'Elective surgeries scheduled for this exact date (specialty mapped to ward)'),
    ('surgeries_scheduled_next_7d', 'Forward-looking: surgeries scheduled over the next 7 days (known in advance, not leakage)'),
    ('safe_staffing_rate_lag1 / safe_staffing_rate_roll7_lag1', 'Lagged safe-staffing-ratio-met rate'),
    ('target_next_day_occupied / target_next_week_occupied', 'Forecast targets: occupied_beds 1 day / 7 days ahead'),
    ('split', 'train / validation / test / future_unscored, assigned by date'),
], columns=['feature', 'description'])

feature_dictionary.to_csv(PROC_DIR / 'feature_dictionary.csv', index=False)
display(feature_dictionary)


,feature,description
0,occupied_beds,Target series: mean hourly occupied beds for t...
1,occupancy_rate,occupied_beds / staffed_beds
2,available_bed_ratio,1 - occupancy_rate
3,closed_bed_ratio,closed_beds / total_beds
4,staffed_bed_ratio,staffed_beds / total_beds
5,hours_ge_85pct / hours_ge_90pct,Share of hours in the day at/above 85%/90% occ...
6,day_of_week / is_weekend / month / day_of_year...,Calendar features
7,dow_sin / dow_cos / month_sin / month_cos,Cyclical encodings of day-of-week and month
8,is_holiday,UK (England & Wales) public holiday indicator
9,occupied_lag_1/7/14/28,Occupied beds 1/7/14/28 days earlier in the sa...


---
## Key Findings

1. **The daily panel is complete and gap-free**: 40 hospital-ward-bed_type series across 721
   days, with no missing dates after aggregation.
2. **Every feature family requested in the brief's technology stack is now represented**: lag
   features, rolling statistics/moving averages, holiday and weekend indicators, seasonal
   (cyclical) features, occupancy ratios, length-of-stay statistics, and bed-utilisation metrics.
3. **The two strongest occupancy drivers from Notebook 02 (admissions, ED arrivals) are now
   engineered features**, not just correlations, ready to be tested for genuine predictive
   value in Notebook 04.
4. **The elective surgery schedule is encoded as a legitimate forward-looking feature**, since
   it is planned in advance; this is explicitly distinguished from, and validated against,
   leakage in Section 12.
5. **The sparse length-of-stay signal (2-9 admissions/day in the smallest wards) required a
   forgiving rolling window plus a documented fallback**, rather than being left to silently
   produce excess missing values.
6. **A supplementary hourly ED-arrivals panel was built** to satisfy the brief's hourly
   forecasting objective where hourly granularity is actually informative, without forcing an
   uninformative hourly grain onto inpatient occupancy modelling (per the Notebook 02 finding
   that hour-of-day occupancy is nearly flat).
7. **All four leakage checks passed**: lag/rolling features are strictly backward-looking, the
   forward-looking surgery feature behaves exactly as intended, and forecast targets are
   correctly aligned.

## Summary

This notebook converted the cleaned datasets and Notebook 02's findings into a single,
leakage-checked, model-ready daily feature panel (plus a supplementary hourly ED panel),
directly addressing the brief's feature-engineering and time-series-preparation scope. Every
engineered feature traces back to either a specific Notebook 02 finding or a specific
feature-engineering item named in the project's technology stack.

## Next Steps (-> Notebook 04: Forecasting Model Development)

1. Train and compare the shortlisted forecasting approaches (SARIMA/SARIMAX, Prophet, XGBoost,
   LightGBM, CatBoost, Random Forest, LSTM) on the `train` split, using `validation` for model
   selection.
2. Apply model-appropriate categorical encoding per Section 10 (native categorical handling for
   tree models; explicit encoding for statistical/deep-learning models).
3. Evaluate both the next-day and next-week targets separately, since they represent different
   forecast horizons with potentially different best-performing models.
4. Reserve the `test` split strictly for final, one-time evaluation in Notebook 05.
5. Carry the `hourly_ed_arrivals_panel` forward for any hourly-forecasting component of the
   final model comparison.
